# 🟢 cash — live feature tour

[cash](https://github.com/galgtonold/cash) caches your notebook **one statement
at a time** and tracks how cells depend on each other, so re-running only
recomputes what actually changed — nothing else.

This tour runs a small Monte Carlo stress test: one shared simulation of the
market, three scenarios stressed off it, one summary. It is real arithmetic —
half a billion random draws in the shared stage — not a `sleep` standing in for
work. About twenty seconds the first time through, and near-instant after that.

**How to use this notebook**

1. **Run all.** Every cell runs once and prints a cash **badge** saying what it
   did (`EXECUTED`, in ochre).
2. **Run all a _second_ time.** The slow cells come back in hundredths of a
   second, each badge carrying a green `cached` chip and a `saved` figure.
3. **Edit one scenario's `SHOCK`** (§3) and Run all — only that scenario pays
   again. The other two, and the shared simulation, stay cached.
4. **Edit `N_STEPS`** (§4) and run *just the summary cell* — cash re-derives the
   whole chain for you.
5. **Edit the `price` helper** (§5) — every cached result that calls it updates.
6. **Restart the kernel** and Run all — the cache survives on disk.

> ### ⚠️ Save the notebook before you run
>
> Step 4 edits one cell and then runs a **different** one. cash reads a cell it
> didn't execute from the saved `.ipynb` file, so an edit still sitting unsaved
> there is invisible to it: it reads the old value, concludes nothing upstream
> changed, and hands you the previous answer while your screen shows the new
> code.
>
> **In JupyterLab (this Binder) and VS Code: press `Ctrl+S` / `Cmd+S` after
> editing, then run.** Autosave exists, but it runs on a timer that is usually
> slower than you are. On Google Colab there is nothing to save — cash reads
> cells live from the frontend.


In [ ]:
# Install cash from PyPI. The [pandas] extra pulls pandas + pyarrow; numpy,
# pandas and matplotlib are already present on Colab. (On a re-run this is a
# fast no-op — pip sees it's already satisfied.)
%pip install -q "cash-lib[pandas]"

import cash
import numpy as np
import pandas as pd

%cash_on

## 1 · A simulation that caches itself

Two functions. That is the whole engine.

`simulate` walks a bundle of price paths forward one day at a time — that loop
is where the seconds go, and it is genuine arithmetic: 2 million paths over 252
trading days is half a billion random draws. `price` turns the paths it
produces into one number.

The shape matters more than the finance. **Slow to compute, small to store** is
exactly the case caching is for: cash keeps the answer, not the work. The
opposite — a fast operation with an enormous result — is the case where it does
not pay, and the badge's `overhead` row will tell you when you have one.


In [ ]:
def simulate(n_paths, n_steps, vol, seed=0):
    """Walk `n_paths` price paths forward `n_steps` days. Returns where they end."""
    rng = np.random.default_rng(seed)
    prices = np.full(n_paths, 100.0)
    drift, scale = -0.5 * vol**2 / n_steps, vol / np.sqrt(n_steps)
    for _ in range(n_steps):
        prices *= np.exp(drift + scale * rng.standard_normal(n_paths))
    return prices


def price(paths, strike):
    """What a call option on those paths is worth, on average."""
    return float(np.maximum(paths - strike, 0.0).mean())


The shared stage. Every scenario below is measured against it, so it is the one
cell the whole notebook hangs from — and the slowest, at roughly eight seconds.


In [ ]:
# ⚙️  Shared settings. §4 changes N_STEPS; everything below is built on these.
N_PATHS  = 2_000_000
N_STEPS  = 252          # trading days in a year
BASE_VOL = 0.20

market   = simulate(N_PATHS, N_STEPS, BASE_VOL)
baseline = price(market, strike=100.0)

print(f"{N_PATHS:,} paths × {N_STEPS} days   |   baseline option value {baseline:,.3f}")


## 2 · Three scenarios, priced independently

Each cell below re-simulates the same book under a different volatility shock
and reports the loss against the baseline. They share the baseline, and nothing
else — which is the point of the next section.

Each takes about two seconds. The shock multiplier sits at the top of its own
cell, so you can edit one without touching the others.


In [ ]:
SHOCK_MILD = 1.25
mild = price(simulate(N_PATHS, 63, BASE_VOL * SHOCK_MILD, seed=1), 100.0) - baseline
print(f"mild   ×{SHOCK_MILD}   {mild:+.3f}")


In [ ]:
SHOCK_SEVERE = 1.60
severe = price(simulate(N_PATHS, 63, BASE_VOL * SHOCK_SEVERE, seed=2), 100.0) - baseline
print(f"severe ×{SHOCK_SEVERE}   {severe:+.3f}")


In [ ]:
# 👇  EDIT THIS NUMBER for §3 — try 2.5. Only this scenario recomputes.
SHOCK_CRASH = 2.00
crash = price(simulate(N_PATHS, 63, BASE_VOL * SHOCK_CRASH, seed=3), 100.0) - baseline
print(f"crash  ×{SHOCK_CRASH}   {crash:+.3f}")


The summary reads all three, so it is the cell that shows any of them changing.


In [ ]:
book = {"mild": mild, "severe": severe, "crash": crash}
worst = max(book, key=lambda k: book[k])

print(f"baseline {baseline:,.3f}")
for name, loss in book.items():
    print(f"  {name:<7}{loss:+8.3f}")
print(f"\nworst case: {worst}  ({book[worst]:+.3f})  at {N_STEPS} days")


## 3 · Run it again — and it's free

Hit **Run all** a second time. The four slow cells finish in hundredths of a
second. Each badge now carries a green `cached` chip and a `saved` figure: cash
recognised that neither the code nor its inputs changed and handed back the
stored answers.

Click a badge open and expand the row for a simulation — it reads `CACHED`,
restored from `RAM`, because it is the same kernel and the same session, so cash
never has to touch disk. The cheap statements beside it — the `print`s — ran as
normal, which is why the header still says `EXECUTED`. **cash caches per
statement, not per cell.** What it skipped was the arithmetic, and the
arithmetic was all of the time.


## 4 · Change one scenario — the others stay cached

This is the section the rest of the notebook exists to set up.

Scroll up to the cell marked 👇 — `SHOCK_CRASH = 2.00` — change it to `2.5` and
hit **Run all**. Then read the badges on the way back down.

The crash scenario pays: its badge has no green `cached` chip and no `saved`
figure, because the number it depends on changed. What did **not** happen is the
interesting half:

* the mild scenario did not re-simulate — green `cached`, a `saved` figure,
  hundredths of a second;
* the severe scenario did not either — the same;
* **the shared market simulation, the expensive one, did not re-run at all.**

One scenario paid; the other two and the eight-second stage all three are built
on did not. From the outside they look like a single block of work, and
recomputing the lot would have been the easy answer. cash tracks what each
statement actually reads, and worked out that exactly one of the three depends
on the number you changed.

The summary redraws, as it always does, because it reads all three.


## 5 · Change the shared setting — then ask for just the answer

§4 ran the lattice one way; this runs it the other.

Go back up to the ⚙️ cell and change `N_STEPS` — try `126`, half a year — then
**save the notebook** (`Ctrl+S` / `Cmd+S`) and run **only the summary cell**
above. Not the cells in between.

Everything descends from that setting, so everything is stale: the market
simulation, all three scenarios, the summary. You asked for none of them. cash
re-derives the chain for you and hands you the answer — you changed one setting,
asked one question, and it worked out the minimum it had to redo.

> **If the number doesn't change, you skipped the save.** cash reads the cell you
> edited but didn't run from the file on disk, so an unsaved `N_STEPS` is still
> the old `N_STEPS` as far as it can tell. (Not applicable on Colab, where cash
> reads cells live.)


## 6 · Edit a helper — everything that calls it updates

cash hashes a function's source **and** the source of the functions it calls. So
when you change an inner helper, every cached result that reaches it — directly
or transitively — invalidates on its own.

`price` is that helper. Change the strike it defaults to, or swap the payoff for
`np.maximum(strike - paths, 0.0)` to price a put instead — then **save** and
**Run all**.

All four results update, because all four call it. You never touched a scenario
cell. And note what *doesn't* move: `simulate` never calls `price`, so the market
paths themselves stay cached — cash follows the call graph rather than throwing
out everything below the line you edited.


## 7 · Beyond notebooks — the `@cash.cache` decorator

Outside cells, wrap any function with `@cash.cache` and it caches by its
arguments and its own source code. The first call runs; an identical call
returns instantly.

Note the `# @cash:assume-safe` on the `time.sleep` line. cash's analyzer flags
calls that look like side effects — a `sleep` throws its return away, which is
exactly the shape of something that matters and would be skipped on a cache hit.
Here it is deliberate, so the line is waived and the warning goes away.

The waiver is **per line**, on purpose. `@cash.cache(assume_safe=True)` would
silence the whole function including anything added to it next year; annotating
the one line you audited means a new unaudited call still speaks up.


In [ ]:
import time

@cash.cache
def var_at(paths, quantile):
    time.sleep(1.0)  # @cash:assume-safe — the sleep IS the stand-in for slow work
    return float(np.quantile(paths, quantile).round(3))

print("first call (runs ~1s):")
print(var_at(market, 0.05))
print("\nsecond call (instant, from cache):")
print(var_at(market, 0.05))


## 8 · It survives a kernel restart

The cache lives on disk, not just in memory. Try **restarting the kernel**, then
**Run all**: the market simulation and all three scenarios restore from cache
instead of recomputing — a fresh kernel picks up right where you left off.


## What did cash save you?

Cache hits, misses, and what cash measured this session:


In [ ]:
%cash_stats


### How to read that

Two of those numbers are **measured**, and they are the pair to compare between
your first run and this one:

- **Compute time** — how long your code actually ran. Around twenty seconds on
  the first pass; a fraction of that once the cache is warm.
- **Statements restored** — results handed back instead of recomputed. Zero on
  the first run, and most of the expensive ones afterwards.

**Net time saved** is deliberately the most pessimistic figure cash can defend.
It credits only savings it re-measured *this session* — and on a clean **Run
all** a statement either restores from cache or computes, never both, so there
is usually nothing to re-measure. That is why it can show a negative lower bound
on the very run that saved you the most. The `at best` end credits each value
with what it cost when first cached.

cash would rather understate a win than claim one it cannot prove. Compute time
is the number that needs no such caveat.
